In [ ]:
# ================================================================================================
# Purchase Prediction Pipeline — Final (Improved)
# ================================================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import OneHotEncoder, RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics         import classification_report, roc_auc_score, f1_score
from scipy.sparse            import hstack
import xgboost as xgb


# ================================================================================================
# 1. Load Data
# ================================================================================================

train = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/train.csv', sep='|')
items = pd.read_csv('/Users/annageiser/Documents/GITHUB/analytics-project/data/raw/items.csv', sep='|')

df = train.merge(items, on='pid', how='left', validate='m:1')
if df[['manufacturer', 'group', 'category']].isna().all().any():
    print("WARNING: Some item columns are entirely missing after merge.")

# ================================================================================================
# 2. Sort
# ================================================================================================

df = df.sort_values(['day', 'lineID']).reset_index(drop=True)

# ================================================================================================
# 3. Sampling  (use 50 % for development; remove for final run)
# ================================================================================================

df_sample = df.iloc[: int(len(df) * 0.1)].copy()

# ================================================================================================
# 4. Feature Engineering (applied before splitting)
# ================================================================================================

df_sample['priceRatio'] = (df_sample['price'] / df_sample['rrp'].replace(0, np.nan)).fillna(1.0)
df_sample['missingCompetitorPrice'] = df_sample['competitorPrice'].isnull().astype(int)

df_sample['weekDay_raw'] = (df_sample['day'] % 7).replace({0: 7})
df_sample['weekDay_sin'] = np.sin(2 * np.pi * df_sample['weekDay_raw'] / 7)
df_sample['weekDay_cos'] = np.cos(2 * np.pi * df_sample['weekDay_raw'] / 7)
df_sample.drop(columns=['weekDay_raw'], inplace=True)

df_sample['priceVsCompetitor'] = (
    df_sample['price'] / df_sample['competitorPrice'].replace(0, np.nan)
).fillna(1.0)                    # missing -> ratio 1.0, flag mitigates

df_sample['priceDiscount'] = (df_sample['rrp'] - df_sample['price']).clip(lower=0)

# Interaction features
df_sample['adFlag_x_priceRatio'] = df_sample['adFlag'] * df_sample['priceRatio']
df_sample['avail_x_priceRatio']  = df_sample['availability'] * df_sample['priceRatio']
df_sample['regulated_generic']   = df_sample['salesIndex'] * df_sample['genericProduct']

# ================================================================================================
# 5. Temporal 80 / 20 Train / Test Split
# ================================================================================================

X = df_sample.drop(columns=['order'])
y = df_sample['order']

split_idx   = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")

# ================================================================================================
# 6. Feature Configuration
# ================================================================================================

NUM_FEATURES = [
    'competitorPrice',
    'priceRatio',
    'priceVsCompetitor',
    'priceDiscount',
    'missingCompetitorPrice',
    'weekDay_sin',
    'weekDay_cos',
    'pid_order_rate',       # rolling product-level historical order rate
    'pid_click_rate',       # rolling product-level historical click rate
    'pid_basket_rate',      # rolling product-level historical basket rate
    'manufacturer_te',      # target-encoded manufacturer
    'group_te',             # target-encoded product group
    'category_te',          # target-encoded category
    'adFlag_x_priceRatio',  # interaction: campaign flag × relative price
    'avail_x_priceRatio',   # interaction: availability × relative price
    'regulated_generic',    # interaction: dispensing regulation × generic status
    'pid_price_std',        # product price volatility over the training window
    'pid_seen_in_train',    # binary: product appeared in training window
]

CAT_FEATURES = [
    'adFlag',
    'availability',
    'content',
    'unit',
    'pharmForm',
    'genericProduct',
    'salesIndex',
]

# These columns represent leakage (or identifiers) and must be removed before modelling.
# click and basket are used only to compute historical behavioural rates.
LEAKAGE_COLS = ['click', 'basket', 'revenue', 'price', 'rrp',
                'lineID', 'day', 'campaignIndex', 'pid']

# Target encoding columns – original categoricals are dropped after encoding.
TE_COLS = [
    ('manufacturer', 'manufacturer_te'),
    ('group',        'group_te'),
    ('category',     'category_te'),
]

# --------------------------------------------------------------------------------------------
# Helper: extract named feature importances from a fitted tree‑based model
# --------------------------------------------------------------------------------------------
def get_feature_importance(model, model_name, num_cols, cat_cols, encoder):
    all_names = list(num_cols) + list(encoder.get_feature_names_out(cat_cols))
    scores = model.feature_importances_
    imp = pd.Series(scores, index=all_names).sort_values(ascending=False)
    return imp


# ================================================================================================
# 7. Rolling-Window Time‑Series Cross‑Validation  (with hyperparameter tuning & window comparison)
# ================================================================================================

WINDOW_LIST = [7, 14, 21]          # experiment with different training windows
cv_results_list = []                # collect per‑window results

for TRAIN_WINDOW in WINDOW_LIST:

    VAL_WINDOW = 1
    print(f"\n{'='*70}\nRunning CV with TRAIN_WINDOW = {TRAIN_WINDOW}\n{'='*70}")

    unique_days  = sorted(X_train['day'].unique())
    fold_results = []

    for i in range(TRAIN_WINDOW, len(unique_days) - VAL_WINDOW + 1):

        train_days = unique_days[i - TRAIN_WINDOW : i]
        val_days   = unique_days[i : i + VAL_WINDOW]

        train_idx = X_train[X_train['day'].isin(train_days)].index
        val_idx   = X_train[X_train['day'].isin(val_days)].index

        X_fold_train = X_train.loc[train_idx].copy()
        X_fold_val   = X_train.loc[val_idx].copy()
        y_fold_train = y_train.loc[train_idx].copy()
        y_fold_val   = y_train.loc[val_idx].copy()

        if len(X_fold_train) == 0 or len(X_fold_val) == 0:
            continue

        # ── 7a. Temporal features (anti‑leakage) ──────────────────────────────────────────────
        fold_labels  = pd.concat([X_fold_train[['pid']], y_fold_train], axis=1)
        pid_rate     = fold_labels.groupby('pid')['order'].mean().rename('pid_order_rate')
        X_fold_train = X_fold_train.join(pid_rate, on='pid')
        X_fold_val   = X_fold_val.join(  pid_rate, on='pid')

        for beh_col, rate_name in [('click', 'pid_click_rate'), ('basket', 'pid_basket_rate')]:
            beh_rate = X_fold_train.groupby('pid')[beh_col].mean().rename(rate_name)
            X_fold_train = X_fold_train.join(beh_rate, on='pid')
            X_fold_val   = X_fold_val.join(  beh_rate, on='pid')

        price_stats = (
            X_fold_train.groupby('pid')['price']
            .agg(['std'])
            .rename(columns={'std': 'pid_price_std'})
        )
        X_fold_train = X_fold_train.join(price_stats, on='pid')
        X_fold_val   = X_fold_val.join(  price_stats, on='pid')

        train_pids = set(X_fold_train['pid'].unique())
        X_fold_train['pid_seen_in_train'] = X_fold_train['pid'].isin(train_pids).astype(int)
        X_fold_val['pid_seen_in_train']   = X_fold_val['pid'].isin(  train_pids).astype(int)

        global_rate = y_fold_train.mean()
        for col, te_col in TE_COLS:
            te_temp     = pd.concat([X_fold_train[[col]], y_fold_train], axis=1)
            group_stats = te_temp.groupby(col)['order'].agg(['mean', 'count'])
            smooth      = (
                (group_stats['mean'] * group_stats['count'] + global_rate * 10) /
                (group_stats['count'] + 10)
            ).to_dict()
            X_fold_train[te_col] = X_fold_train[col].map(smooth).fillna(global_rate)
            X_fold_val[  te_col] = X_fold_val[  col].map(smooth).fillna(global_rate)

        # Drop original categorical columns used for TE (clean‑up)
        for col, _ in TE_COLS:
            X_fold_train.drop(columns=[col], errors='ignore', inplace=True)
            X_fold_val.drop(  columns=[col], errors='ignore', inplace=True)

        # ── 7b. Drop leakage / identifier columns ─────────────────────────────────────────────
        X_fold_train.drop(columns=LEAKAGE_COLS, errors='ignore', inplace=True)
        X_fold_val.drop(  columns=LEAKAGE_COLS, errors='ignore', inplace=True)

        # ── 7c. Correlation filter ────────────────────────────────────────────────────────────
        active_num  = [f for f in NUM_FEATURES if f in X_fold_train.columns]
        corr_matrix = X_fold_train[active_num].corr().abs()
        upper_tri   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        corr_drop   = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
        if corr_drop:
            print(f"  Fold {i}: corr-drop {corr_drop}")

        X_fold_train.drop(columns=corr_drop, errors='ignore', inplace=True)
        X_fold_val.drop(  columns=corr_drop, errors='ignore', inplace=True)

        active_num = [f for f in active_num if f not in corr_drop]
        active_cat = [f for f in CAT_FEATURES if f in X_fold_train.columns]

        # ── 7d. Imputation and casting ─────────────────────────────────────────────────────────
        fill_vals = {col: 'Missing' for col in active_cat}
        X_fold_train.fillna(fill_vals, inplace=True)
        X_fold_val.fillna(  fill_vals, inplace=True)
        X_fold_train[active_cat] = X_fold_train[active_cat].astype(str)
        X_fold_val[  active_cat] = X_fold_val[  active_cat].astype(str)

        # ── 7e. Encoding and scaling (RobustScaler) ───────────────────────────────────────────
        imputer = SimpleImputer(strategy='median')
        scaler  = RobustScaler()
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

        X_train_num = scaler.fit_transform(imputer.fit_transform(X_fold_train[active_num]))
        X_val_num   = scaler.transform(    imputer.transform(    X_fold_val[  active_num]))
        X_train_cat = encoder.fit_transform(X_fold_train[active_cat])
        X_val_cat   = encoder.transform(    X_fold_val[  active_cat])

        X_tr_fold = hstack([X_train_num, X_train_cat])
        X_val_fold   = hstack([X_val_num,   X_val_cat])

        # ── 7f. Hyperparameter tuning inside the fold (nested time‑series split) ──────────────
        # Use a 3‑split TimeSeriesSplit on the training fold (rows are sorted by day)
        tscv = TimeSeriesSplit(n_splits=3)

        # Parameter grids
        param_grids = {
            'DecisionTree': {
                'max_depth': [5, 8, 12, None],
                'min_samples_leaf': [10, 50, 100]
            },
            'RandomForest': {
                'n_estimators': [100, 150],
                'max_depth': [8, 10, 12],
                'min_samples_leaf': [20, 50],
            },
            'XGBoost': {
                'n_estimators': [200, 400],
                'max_depth': [4, 6, 8],
                'learning_rate': [0.05, 0.1],
                'subsample': [0.7, 0.8],
                'min_child_weight': [3, 5],
            },
        }

        spw = (y_fold_train == 0).sum() / (y_fold_train == 1).sum()

        models_to_run = {
            'DecisionTree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
            'RandomForest': RandomForestClassifier(class_weight='balanced_subsample', n_jobs=-1, random_state=42),
            'XGBoost': xgb.XGBClassifier(scale_pos_weight=spw, eval_metric='auc',
                                         use_label_encoder=False, random_state=42, verbosity=0),
        }

        fold_line = (f"Fold {i:3d} | val ends day {val_days[-1]:3d} "
                     f"| train {len(X_fold_train):5,} | val {len(X_fold_val):5,}")
        for model_name, base_model in models_to_run.items():
            # Hyperparameter search only if grid is defined
            if model_name in param_grids:
                search = RandomizedSearchCV(
                    base_model, param_grids[model_name],
                    cv=tscv, scoring='roc_auc', n_iter=10,
                    random_state=42, n_jobs=-1
                )
                search.fit(X_tr_fold, y_fold_train)
                best_model = search.best_estimator_
            else:
                best_model = base_model
                best_model.fit(X_tr_fold, y_fold_train)

            y_prob = best_model.predict_proba(X_val_fold)[:, 1]
            y_pred = best_model.predict(X_val_fold)
            auc    = roc_auc_score(y_fold_val, y_prob)
            f1     = f1_score(y_fold_val, y_pred, zero_division=0)
            fold_results.append({
                'model': model_name, 'fold': i, 'val_day': val_days[-1],
                'auc': auc, 'f1': f1,
                'train_size': len(X_fold_train), 'val_size': len(X_fold_val),
                'window': TRAIN_WINDOW
            })
            fold_line += f"  |  {model_name[:5]} AUC {auc:.4f} F1 {f1:.4f}"

            # Show feature importance for the first fold of each window only
            if i == TRAIN_WINDOW:
                try:
                    imp = get_feature_importance(best_model, model_name, active_num, active_cat, encoder)
                    print(f"\n  [{model_name}] Top 10 features:")
                    print(imp.head(10).to_string())
                except:
                    pass

        print(fold_line)

        if i == TRAIN_WINDOW:
            print(f"\n  Numeric     : {active_num}")
            print(f"  Categorical : {active_cat}\n")

    # Store results for this window
    cv_results_list.append(pd.DataFrame(fold_results))

# ── Cross‑validation summary across windows ────────────────────────────────────────────────────
all_cv = pd.concat(cv_results_list, ignore_index=True)
print("\n── CV AUC & F1 by Model and Window ──────────────────────────")
summary = all_cv.groupby(['window', 'model'])[['auc', 'f1']].mean().round(4)
print(summary.to_string())
best_window = all_cv.groupby('window')['auc'].mean().idxmax()
print(f"\nBest window (by mean AUC): {best_window}")


# ================================================================================================
# 8. Final Model — Expanding‑Window Features on Full Training Set, then Evaluate on Test
# ================================================================================================

print("\n── Final Model — Training on full X_train with expanding‑window features ──")

# --------------------------------------------------------------------------------------------
# 8a. Build expanding‑window historical features for the training set (no leakage)
# --------------------------------------------------------------------------------------------
def build_expanding_train_features(df, y_series):
    """
    Computes pid_order_rate, pid_click_rate, pid_basket_rate,
    pid_price_std, pid_seen_in_train, and target encodings
    using only data from days strictly earlier than the current day.
    Returns: df enriched with new columns, and a state dict for test time.
    """
    df = df.copy()
    df = df.sort_values('day').reset_index(drop=True)
    y_series = y_series.loc[df.index]       # align y

    # Initialise cumulative storage (data up to the previous day)
    pid_total = {}
    pid_order = {}
    pid_click = {}
    pid_basket = {}
    pid_prices = {}        # list of all past prices per pid

    # For target encoding groups
    grp_total = {}
    grp_order = {}
    man_total = {}
    man_order = {}
    cat_total = {}
    cat_order = {}

    global_orders = 0
    global_total = 0

    new_cols = ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate',
                'pid_price_std', 'pid_seen_in_train',
                'manufacturer_te', 'group_te', 'category_te']
    for col in new_cols:
        df[col] = np.nan

    unique_days = sorted(df['day'].unique())
    for day in unique_days:
        mask = df['day'] == day
        day_df = df.loc[mask]
        y_day = y_series.loc[mask]

        # Current global rate (from previous days)
        prev_global_rate = global_orders / global_total if global_total > 0 else 0.0

        # ---- pid-level features ----
        for pid_val in day_df['pid'].unique():
            idx = day_df.index[day_df['pid'] == pid_val]

            # rates
            total_pid = pid_total.get(pid_val, 0)
            if total_pid > 0:
                df.loc[idx, 'pid_order_rate']  = pid_order.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_click_rate']  = pid_click.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_basket_rate'] = pid_basket.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_seen_in_train'] = 1
            else:
                df.loc[idx, 'pid_order_rate']  = prev_global_rate
                df.loc[idx, 'pid_click_rate']  = prev_global_rate   # rough proxy
                df.loc[idx, 'pid_basket_rate'] = prev_global_rate
                df.loc[idx, 'pid_seen_in_train'] = 0

            # price volatility
            past_prices = pid_prices.get(pid_val, [])
            if len(past_prices) >= 2:
                df.loc[idx, 'pid_price_std'] = np.std(past_prices, ddof=0)
            else:
                df.loc[idx, 'pid_price_std'] = 0.0

        # ---- target encoding features ----
        for col_orig, col_te, order_dict, total_dict in [
            ('manufacturer', 'manufacturer_te', man_order, man_total),
            ('group',        'group_te',         grp_order,  grp_total),
            ('category',     'category_te',      cat_order,  cat_total)]:
            for val in day_df[col_orig].unique():
                idx_val = day_df.index[day_df[col_orig] == val]
                n_orders = order_dict.get(val, 0)
                n_total  = total_dict.get(val, 0)
                smoothing = 10
                te = (n_orders + prev_global_rate * smoothing) / (n_total + smoothing) if (n_total + smoothing) > 0 else prev_global_rate
                df.loc[idx_val, col_te] = te

        # ---- Update cumulative statistics for the next days ----
        # pid counts and orders
        pid_day_counts = day_df.groupby('pid').size()
        pid_day_orders = y_day.groupby(day_df['pid']).sum()
        for pid_val in pid_day_counts.index:
            pid_total[pid_val] = pid_total.get(pid_val, 0) + pid_day_counts[pid_val]
            pid_order[pid_val] = pid_order.get(pid_val, 0) + pid_day_orders.get(pid_val, 0)
        # clicks and baskets
        for beh_col, cum_dict in [('click', pid_click), ('basket', pid_basket)]:
            day_sum = day_df.groupby('pid')[beh_col].sum()
            for pid_val in day_sum.index:
                cum_dict[pid_val] = cum_dict.get(pid_val, 0) + day_sum[pid_val]

        # prices: append all day prices
        for pid_val, prices in day_df.groupby('pid')['price'].apply(list).items():
            if pid_val not in pid_prices:
                pid_prices[pid_val] = []
            pid_prices[pid_val].extend(prices)

        # group counts and orders
        for col_orig, order_dict, total_dict in [
            ('manufacturer', man_order, man_total),
            ('group',        grp_order,  grp_total),
            ('category',     cat_order,  cat_total)]:
            day_grp = day_df.groupby(col_orig)
            day_cnt = day_grp.size()
            day_ord = y_day.groupby(day_df[col_orig]).sum()
            for val in day_cnt.index:
                total_dict[val] = total_dict.get(val, 0) + day_cnt[val]
                order_dict[val] = order_dict.get(val, 0) + day_ord.get(val, 0)

        # global totals
        global_total += len(day_df)
        global_orders += y_day.sum()

    return df


# --------------------------------------------------------------------------------------------
# 8b. Build features for the test set using final training state
# --------------------------------------------------------------------------------------------
def build_test_features(test_df, train_df_with_features):
    """
    Uses the final cumulative state from the training set (implicit in the features
    of the last training day) to assign exact same statistics to the test set.
    For simplicity we extract the final day's features per pid and group.
    """
    # Last day in training
    last_train_day = train_df_with_features['day'].max()
    last_train = train_df_with_features[train_df_with_features['day'] == last_train_day]

    # pid-level features: just take the last observed values for each pid
    pid_feats = last_train.groupby('pid')[['pid_order_rate', 'pid_click_rate',
                                          'pid_basket_rate', 'pid_price_std',
                                          'pid_seen_in_train']].last()
    # For pids not seen in training, fill with global final rate
    global_final_rate = last_train['pid_order_rate'].mean()  # rough
    test_df = test_df.join(pid_feats, on='pid')
    for col in ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate', 'pid_price_std']:
        test_df[col] = test_df[col].fillna(global_final_rate if col != 'pid_price_std' else 0.0)
    test_df['pid_seen_in_train'] = test_df['pid_seen_in_train'].fillna(0).astype(int)

    # TE features: use last training day's map per group (already final)
    for col_orig, col_te in [('manufacturer', 'manufacturer_te'),
                             ('group', 'group_te'),
                             ('category', 'category_te')]:
        te_map = last_train.groupby(col_orig)[col_te].last()
        test_df[col_te] = test_df[col_orig].map(te_map).fillna(global_final_rate)

    return test_df


# ── 8c. Prepare X_train and X_test with expanding‑window features ──
X_train_exp = build_expanding_train_features(X_train.copy(), y_train)
X_test_exp  = build_test_features(X_test.copy(), X_train_exp)

# Drop original categorical columns used for TE
for col, _ in TE_COLS:
    X_train_exp.drop(columns=[col], errors='ignore', inplace=True)
    X_test_exp.drop( columns=[col], errors='ignore', inplace=True)

# Drop leakage columns
X_train_exp.drop(columns=LEAKAGE_COLS, errors='ignore', inplace=True)
X_test_exp.drop( columns=LEAKAGE_COLS, errors='ignore', inplace=True)

# Correlation filter
active_num_exp = [f for f in NUM_FEATURES if f in X_train_exp.columns]
corr_exp = X_train_exp[active_num_exp].corr().abs()
upper_exp = corr_exp.where(np.triu(np.ones(corr_exp.shape), k=1).astype(bool))
drop_exp = [col for col in upper_exp.columns if any(upper_exp[col] > 0.95)]
if drop_exp:
    print(f"Final model — dropping correlated features: {drop_exp}")
X_train_exp.drop(columns=drop_exp, errors='ignore', inplace=True)
X_test_exp.drop( columns=drop_exp, errors='ignore', inplace=True)
active_num_exp = [f for f in active_num_exp if f not in drop_exp]
active_cat_exp = [f for f in CAT_FEATURES if f in X_train_exp.columns]

# Impute and cast
fill_exp = {col: 'Missing' for col in active_cat_exp}
X_train_exp.fillna(fill_exp, inplace=True)
X_test_exp.fillna( fill_exp, inplace=True)
X_train_exp[active_cat_exp] = X_train_exp[active_cat_exp].astype(str)
X_test_exp[ active_cat_exp] = X_test_exp[ active_cat_exp].astype(str)

# Encode and scale (RobustScaler)
imputer_exp = SimpleImputer(strategy='median')
scaler_exp  = RobustScaler()
encoder_exp = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

X_tr_num = scaler_exp.fit_transform(imputer_exp.fit_transform(X_train_exp[active_num_exp]))
X_te_num = scaler_exp.transform(    imputer_exp.transform(    X_test_exp[ active_num_exp]))
X_tr_cat = encoder_exp.fit_transform(X_train_exp[active_cat_exp])
X_te_cat = encoder_exp.transform(    X_test_exp[ active_cat_exp])

X_tr_final = hstack([X_tr_num, X_tr_cat])
X_te_final = hstack([X_te_num, X_te_cat])

# ── 8d. Final hyperparameter tuning on full training set using time‑series split ──
spw_final = (y_train == 0).sum() / (y_train == 1).sum()
base_xgb = xgb.XGBClassifier(scale_pos_weight=spw_final, eval_metric='auc',
                             use_label_encoder=False, random_state=42, verbosity=0)

xgb_param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.7, 0.8],
    'min_child_weight': [3, 5],
}

tscv_full = TimeSeriesSplit(n_splits=3)
search_final = RandomizedSearchCV(
    base_xgb, xgb_param_grid, cv=tscv_full, scoring='roc_auc',
    n_iter=10, random_state=42, n_jobs=-1
)
search_final.fit(X_tr_final, y_train)
best_xgb = search_final.best_estimator_

# ── 8e. Evaluate final model on test set ─────────────────────────────────────────────────────
y_prob = best_xgb.predict_proba(X_te_final)[:, 1]
y_pred = best_xgb.predict(X_te_final)
auc    = roc_auc_score(y_test, y_prob)
f1     = f1_score(y_test, y_pred, zero_division=0)

print(f"\nFinal XGBoost (tuned)  |  AUC {auc:.4f}  |  F1 {f1:.4f}")
print(classification_report(y_test, y_pred, target_names=['no order', 'order']))

# Feature importance
imp = get_feature_importance(best_xgb, 'XGBoost', active_num_exp, active_cat_exp, encoder_exp)
print(f"\n  Top 15 features (XGBoost):")
print(imp.head(15).to_string())

Train size: 1,102,400  |  Test size: 275,601
Train positive rate: 0.295  |  Test positive rate: 0.203

Running CV with TRAIN_WINDOW = 7

  [DecisionTree] Top 10 features:
pid_order_rate        9.971146e-01
weekDay_cos           2.250184e-03
weekDay_sin           5.831666e-04
avail_x_priceRatio    5.203829e-05
content_1             7.540752e-16
content_2X100         0.000000e+00
content_9.5           0.000000e+00
pharmForm_LIQ         0.000000e+00
pharmForm_SUP         0.000000e+00
pharmForm_Ton         0.000000e+00

  [RandomForest] Top 10 features:
pid_order_rate        0.320204
pid_click_rate        0.159662
pid_basket_rate       0.071521
group_te              0.068378
category_te           0.058244
manufacturer_te       0.049722
priceRatio            0.038177
avail_x_priceRatio    0.037119
competitorPrice       0.019810
priceVsCompetitor     0.018565

  [XGBoost] Top 10 features:
pid_order_rate    0.403401
weekDay_cos       0.012125
pharmForm_TAB     0.011568
content_12        0.011